## 实验2：构造/析构函数

本实验通过输出语句观察对象从创建到销毁的完整生命周期，重点理解构造函数、成员初始化列表、析构函数和词法作用域之间的关系。请先执行头文件单元，再按顺序执行后续代码单元。

### 基础实验代码

In [2]:
#include <iostream>
#include <string>

In [2]:
class User {
public:
    User(
        std::string name,
        int age
    ) {
        name_ = name;
        age_ = age;

        std::cout
            << "User constructor\n";
    }

    void print() const {
        std::cout
            << "name = " << name_
            << ", age = " << age_
            << '\n';
    }

private:
    std::string name_;
    int age_;
};

{
    User user("Bob", 20);

    user.print();
}

User constructor
name = Bob, age = 20


### 构造函数真正解决的问题
将构造函数只理解成“创建对象时自动执行的函数”还不够。
更重要的是：**构造函数负责建立对象的初始状态，使对象从诞生起就满足类的不变量。**
例如：
```C++
class User{
private:
    std::string name_;
    int age_;
};
```

我们可能会规定：名字不能为空，年龄必须大于0，于是：
```C++
User user("Bob", 20);
```
如果构造函数在写入成员前检查这些条件，那么一旦构造成功，对象就应当处于合法状态。

这种对象在整个可观察生命周期中都必须满足的约束，就是**类的不变量**。后续所有公开成员函数也应维护它。

#### 不要先赋值，使用**初始化列表**
```C++
User(
    std::string name,
    int age
) {
    name_ = name;
    age_ = age;
}
```
上述代码虽然可以工作，但通常不是推荐的形式。成员会先完成默认初始化，然后才在构造函数体内被赋值。

修改为：
```C++
User(
    std::string name,
    int age
)
    : name_(name),
      age_(age) {
}
```

完整的代码片段：
```C++
class User {
public:
    User(
        std::string name,
        int age
    )
        : name_(name),
          age_(age) {
    }

private:
    std::string name_;
    int age_;
};
```
这叫做成员初始化列表。它直接决定成员如何构造，对于引用成员、`const` 成员以及没有默认构造函数的成员更是必需的。

### 构造和赋值不是一回事
理解：
```C++
name_(name);
```
表示使用参数 `name` 直接构造成员 `name_`。

而：
```C++
name_ = name;
```
表示 `name_` 已经先完成构造，进入函数体后再执行赋值。对 `std::string` 等对象来说，这通常多出一次不必要的状态变更。

概念上是：
- 初始化列表
```
初始化列表

内存
 ↓
构造
 ↓
就绪
```

- 构造赋值
```
内存
 ↓
默认构造函数
 ↓
赋值
 ↓
就绪
```

### 析构函数
继续在实验代码的基础上添加析构函数。由于 C++ Jupyter kernel 会保留前面单元格中的类型定义，这里使用新的类名 `UserWithDestructor`，避免与第一段实验的 `User` 重定义冲突：

In [3]:
class UserWithDestructor {
public:
    UserWithDestructor(
        std::string name,
        int age
    )
        : name_(name),
          age_(age) {

        std::cout
            << "User constructor\n";
    }

    ~UserWithDestructor() {
        std::cout
            << "User destructor\n";
    }

    void print() const {
        std::cout
            << "name = " << name_
            << ", age = " << age_
            << '\n';
    }

private:
    std::string name_;
    int age_;
};


{
    std::cout << "before UserWithDestructor\n";

    UserWithDestructor user(
        "Bob",
        20
    );

    user.print();

    std::cout << "after UserWithDestructor\n";
}

before UserWithDestructor
User constructor
name = Bob, age = 20
after UserWithDestructor
User destructor


观察输出：
```
before UserWithDestructor
User constructor
name = Bob, age = 20
after UserWithDestructor
User destructor
```

注意：最后一条析构输出发生在内部对象 `user` 离开所在作用域之后。析构函数不是由代码显式调用，而是由自动存储期对象的生命周期规则触发。真正的流程是：
```
进入 `main()`

 ↓

调用 `UserWithDestructor` 构造函数

 ↓

调用 `user.print()` 输出对象信息

 ↓

输出 `after UserWithDestructor`

 ↓

`main` 函数中的当前作用域结束

 ↓

调用 `UserWithDestructor` 析构函数
```

### 用作用域观察生命周期
修改主函数：

In [5]:
{
    std::cout << "A\n";

    {
        std::cout << "B\n";

        UserWithDestructor user1(
            "Bob",
            20
        );

        std::cout << "C\n";
    }

    std::cout << "D\n";
}

A
B
User constructor
C
User destructor
D


观察运行结果：
```
A
B
User constructor
C
User destructor
D
```
从输出顺序可以得到：

- `user1` 进入内部 `{}` 时构造。
- 执行到内部 `}` 时，`user1` 立即析构，因此析构输出位于 `C` 和 `D` 之间。
- 自动对象不是必须等到整个程序结束才销毁，而是在它所属的作用域结束时销毁。
- 同一作用域存在多个自动对象时，通常按构造顺序的逆序析构，这为 RAII 资源释放奠定了基础。

### 实验结论

构造函数建立合法状态，析构函数在生命周期结束时完成清理；二者共同界定对象管理资源的时间范围。后续 RAII 会利用这一确定性来管理文件、内存、锁和 Native handle。